<a href="https://colab.research.google.com/github/Narashima1808/Text-Summarization-Evaluator/blob/main/Comparative_Evaluation_of_Extractive_vs_Abstractive_Text_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets rouge-score sentencepiece accelerate -q
!pip install nltk bert-score -q


In [ ]:
import os, re, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict
from typing import List, Dict

import torch
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset
from transformers import (
    BartForConditionalGeneration, BartTokenizer,
    T5ForConditionalGeneration, T5Tokenizer
)
from rouge_score import rouge_scorer

nltk.download('punkt',       quiet=True)
nltk.download('punkt_tab',   quiet=True)
nltk.download('stopwords',   quiet=True)
warnings.filterwarnings('ignore')

# ── Global Config ──────────────────────────────────────────
CONFIG = {
    "dataset"         : "cnn_dailymail",
    "dataset_version" : "3.0.0",
    "num_samples"     : 100,
    "extractive_ratio": 0.3,
    "bart_model"      : "facebook/bart-large-cnn",
    "t5_model"        : "t5-small",
    "max_input_len"   : 512,
    "max_output_len"  : 128,
    "min_output_len"  : 30,
    "seed"            : 42,
    "device"          : "cuda" if torch.cuda.is_available() else "cpu"
}

np.random.seed(CONFIG["seed"])

if CONFIG["device"] == "cuda":
    print(f" GPU detected : {torch.cuda.get_device_name(0)}")
else:
    print(" No GPU. Running on CPU — BART/T5 will be slow.")
    print("   Go to: Runtime → Change runtime type → T4 GPU")

print(f"\n Config ready | Dataset: {CONFIG['dataset']} | Samples: {CONFIG['num_samples']}")
print(f"   Device: {CONFIG['device'].upper()} | Seed: {CONFIG['seed']}")

In [ ]:

print(" Loading CNN/DailyMail test split...")
print("   (Downloads ~1 GB on first run, cached after)\n")

raw_dataset = load_dataset("cnn_dailymail", "3.0.0", split="test", trust_remote_code=True)
print(f"   Total test samples: {len(raw_dataset)}")

# ── Sample ────────────────────────────────────────────────
rng     = np.random.default_rng(CONFIG["seed"])
indices = rng.choice(len(raw_dataset), CONFIG["num_samples"], replace=False).tolist()
samples = raw_dataset.select(indices)

articles  = [s["article"]    for s in samples]
summaries = [s["highlights"] for s in samples]

print(f" Sampled {len(articles)} articles.\n")

# ── Statistics ────────────────────────────────────────────
art_lens  = [len(a.split()) for a in articles]
sum_lens  = [len(s.split()) for s in summaries]
comp_gt   = [sl / max(al, 1) for al, sl in zip(art_lens, sum_lens)]

print("=" * 58)
print("           EXPLORATORY DATA ANALYSIS")
print("=" * 58)
print(f"  Articles  — Avg: {np.mean(art_lens):.0f} | Med: {np.median(art_lens):.0f} | Max: {max(art_lens)}")
print(f"  Summaries — Avg: {np.mean(sum_lens):.0f} | Med: {np.median(sum_lens):.0f} | Max: {max(sum_lens)}")
print(f"  Compression ratio — Mean: {np.mean(comp_gt):.4f} ({np.mean(comp_gt)*100:.1f}% of article)")
print("=" * 58)

# ── Plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"CNN/DailyMail — Dataset Statistics  (N={CONFIG['num_samples']})",
             fontsize=13, fontweight='bold')

axes[0].hist(art_lens, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(art_lens), color='red', linestyle='--', linewidth=1.3,
                label=f"Mean={np.mean(art_lens):.0f}")
axes[0].set_title("Article Length (words)"); axes[0].set_xlabel("Words")
axes[0].set_ylabel("Frequency"); axes[0].legend(fontsize=8)

axes[1].hist(sum_lens, bins=30, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(np.mean(sum_lens), color='navy', linestyle='--', linewidth=1.3,
                label=f"Mean={np.mean(sum_lens):.0f}")
axes[1].set_title("Summary Length (words)"); axes[1].set_xlabel("Words")
axes[1].legend(fontsize=8)

axes[2].hist(comp_gt, bins=30, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[2].axvline(np.mean(comp_gt), color='darkred', linestyle='--', linewidth=1.3,
                label=f"Mean={np.mean(comp_gt):.3f}")
axes[2].set_title("Compression Ratio (GT)"); axes[2].set_xlabel("Ratio")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig("eda_stats.png", dpi=150, bbox_inches='tight')
plt.show()
print(" Saved: eda_stats.png")

# ── Quick sanity check ────────────────────────────────────
print(f"\n── Sample 0 preview ──")
print(f"Article  : {articles[0][:200]}...")
print(f"Reference: {summaries[0]}")

In [ ]:

# TF-IDF EXTRACTIVE SUMMARIZATION
#
# Method:
#   1. Tokenize document into sentences
#   2. Build TF-IDF matrix — each sentence = one doc
#   3. Score each sentence = mean TF-IDF weight of its tokens
#   4. Keep top-K sentences (K = ratio × total), in original order
#
# Properties:
#    100% faithful — no words added outside source
#    Extremely fast (no model needed)
#    Sentences may be disjointed / incoherent
#    Cannot paraphrase, merge, or generalise
# ============================================================

class TFIDFSummarizer:
    def __init__(self, ratio: float = 0.3, min_sents: int = 2):
        self.ratio     = ratio
        self.min_sents = min_sents
        self.stopwords = set(stopwords.words('english'))

    def _clean(self, text: str) -> str:
        text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
        return ' '.join(w for w in text.split() if w not in self.stopwords)

    def summarize(self, text: str) -> str:
        sentences = sent_tokenize(text)
        if len(sentences) <= self.min_sents:
            return text

        cleaned = [self._clean(s) for s in sentences]
        # Remove fully empty strings (can crash TfidfVectorizer)
        cleaned = [c if c.strip() else 'empty' for c in cleaned]

        try:
            vectorizer   = TfidfVectorizer()
            tfidf_matrix = vectorizer.fit_transform(cleaned)
        except Exception:
            return ' '.join(sentences[:self.min_sents])

        scores      = np.array(tfidf_matrix.mean(axis=1)).flatten()
        k           = max(self.min_sents, int(len(sentences) * self.ratio))
        top_indices = sorted(np.argsort(scores)[-k:])
        return ' '.join(sentences[i] for i in top_indices)


# ── Run ───────────────────────────────────────────────────
print("  Running TF-IDF Extractive Summarizer...")

tfidf_model      = TFIDFSummarizer(ratio=CONFIG["extractive_ratio"])
t_start          = time.time()
tfidf_summaries  = [tfidf_model.summarize(a) for a in articles]
tfidf_time       = time.time() - t_start

print(f" Done in {tfidf_time:.2f}s  ({tfidf_time/len(articles)*1000:.1f} ms/sample)\n")
print("── Sample output ──")
print(f"TF-IDF : {tfidf_summaries[0][:300]}...")
print(f"Ref    : {summaries[0]}")

In [ ]:

# BART ABSTRACTIVE SUMMARIZATION
#
# Model: facebook/bart-large-cnn  (~1.6 GB)
# Architecture:
#   Encoder: Bidirectional transformer (like BERT)
#   Decoder: Left-to-right autoregressive transformer (like GPT)
#   Pre-training: Denoising (text corruption + reconstruction)
#   Fine-tuning:  CNN/DailyMail summarization
#
# Generation strategy: Beam search (4 beams), length penalty
#
# Properties:
#    Fluent, coherent output
#    Best ROUGE on CNN/DM (fine-tuned on same distribution)
#    Risk of hallucination (generates unseen facts)
#    Slow without GPU


print("📥 Loading BART (facebook/bart-large-cnn)...")
print("   ~1.6 GB download on first run\n")

bart_tokenizer = BartTokenizer.from_pretrained(CONFIG["bart_model"])
bart_model     = BartForConditionalGeneration.from_pretrained(CONFIG["bart_model"])
bart_model     = bart_model.to(CONFIG["device"])
bart_model.eval()
print(f" BART loaded on {CONFIG['device'].upper()}\n")


def bart_summarize(text: str) -> str:
    inputs = bart_tokenizer(
        text,
        return_tensors="pt",
        max_length=CONFIG["max_input_len"],
        truncation=True
    ).to(CONFIG["device"])

    with torch.no_grad():
        ids = bart_model.generate(
            inputs["input_ids"],
            max_length        = CONFIG["max_output_len"],
            min_length        = CONFIG["min_output_len"],
            num_beams         = 4,
            length_penalty    = 2.0,
            early_stopping    = True,
            no_repeat_ngram_size = 3
        )
    return bart_tokenizer.decode(ids[0], skip_special_tokens=True)


# ── Run ───────────────────────────────────────────────────
print(f"  Generating BART summaries ({len(articles)} samples)...")
print("   GPU: ~2–4 min | CPU: ~30–60 min\n")

bart_summaries = []
t_start = time.time()

for i, art in enumerate(articles):
    bart_summaries.append(bart_summarize(art))
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t_start
        eta     = elapsed / (i + 1) * (len(articles) - i - 1)
        print(f"   [{i+1:3d}/{len(articles)}]  elapsed: {elapsed:.0f}s  ETA: {eta:.0f}s")

bart_time = time.time() - t_start
print(f"\n BART done in {bart_time:.1f}s  ({bart_time/len(articles):.2f}s/sample)\n")
print(f"BART sample: {bart_summaries[0]}")

In [ ]:
CONFIG = {
    "dataset"         : "cnn_dailymail",
    "dataset_version" : "3.0.0",
    "num_samples"     : 100,
    "extractive_ratio": 0.3,
    "bart_model"      : "facebook/bart-large-cnn",
    "t5_model"        : "t5-small",
    "max_input_len"   : 512,
    "max_output_len"  : 128,
    "min_output_len"  : 30,
    "seed"            : 42,
    "device"          : "cuda" if torch.cuda.is_available() else "cpu"
}


# T5 ABSTRACTIVE SUMMARIZATION
#
# Model: t5-small (60M params) — swap to t5-base (220M) for
#        better quality if you have >12 GB RAM
#
# Key difference from BART:
#   T5 treats EVERY NLP task as text-to-text. The task is
#   specified via a prefix: "summarize: <article>"
#   Trained on C4 (Colossal Clean Crawled Corpus) — broader
#   but less specialised than BART on news.
#
# Properties:
#    Good generalisation across domains
#    Faster than BART (t5-small is tiny)
#   x Lower ROUGE on CNN/DM vs BART (not fine-tuned on it)
#   x Occasional repetition at small model sizes


from transformers import T5ForConditionalGeneration, T5Tokenizer

print("📥 Loading T5 (t5-small)...")

t5_tokenizer = T5Tokenizer.from_pretrained(CONFIG["t5_model"], legacy=False)
t5_model     = T5ForConditionalGeneration.from_pretrained(CONFIG["t5_model"])
t5_model     = t5_model.to(CONFIG["device"])
t5_model.eval()
print(f" T5 loaded on {CONFIG['device'].upper()}\n")


def t5_summarize(text: str) -> str:
    # T5 REQUIRES the "summarize: " prefix — without it output degrades
    input_text = "summarize: " + text

    inputs = t5_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=CONFIG["max_input_len"],
        truncation=True
    ).to(CONFIG["device"])

    with torch.no_grad():
        ids = t5_model.generate(
            inputs["input_ids"],
            max_length        = CONFIG["max_output_len"],
            min_length        = CONFIG["min_output_len"],
            num_beams         = 4,
            length_penalty    = 2.0,
            early_stopping    = True,
            no_repeat_ngram_size = 3
        )
    return t5_tokenizer.decode(ids[0], skip_special_tokens=True)


# ── Run ───────────────────────────────────────────────────
print(f"  Generating T5 summaries ({len(articles)} samples)...")

t5_summaries = []
t_start = time.time()

for i, art in enumerate(articles):
    t5_summaries.append(t5_summarize(art))
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t_start
        eta     = elapsed / (i + 1) * (len(articles) - i - 1)
        print(f"   [{i+1:3d}/{len(articles)}]  elapsed: {elapsed:.0f}s  ETA: {eta:.0f}s")

t5_time = time.time() - t_start
print(f"\n T5 done in {t5_time:.1f}s  ({t5_time/len(articles):.2f}s/sample)\n")
print(f"T5 sample: {t5_summaries[0]}")

In [ ]:

# ROUGE EVALUATION
#
# ROUGE-1 : unigram overlap  → content coverage
# ROUGE-2 : bigram overlap   → phrase-level fluency
# ROUGE-L : longest common subsequence → sentence structure
#
# We report F1 (primary), Precision, and Recall.
# use_stemmer=True normalises word forms (run/running → run)


scorer_obj = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
)

def compute_rouge(hypotheses: List[str], references: List[str]) -> Dict:
    agg = defaultdict(lambda: {'p': [], 'r': [], 'f': []})
    for hyp, ref in zip(hypotheses, references):
        scores = scorer_obj.score(ref, hyp)
        for m, v in scores.items():
            agg[m]['p'].append(v.precision)
            agg[m]['r'].append(v.recall)
            agg[m]['f'].append(v.fmeasure)
    return {
        m: {
            'precision': np.mean(vals['p']),
            'recall'   : np.mean(vals['r']),
            'f1'       : np.mean(vals['f'])
        }
        for m, vals in agg.items()
    }


print(" Computing ROUGE scores for all 3 models...\n")

results = {
    "TF-IDF (Extractive)" : compute_rouge(tfidf_summaries, summaries),
    "BART (Abstractive)"  : compute_rouge(bart_summaries,  summaries),
    "T5 (Abstractive)"    : compute_rouge(t5_summaries,    summaries),
}

# ── Print table ───────────────────────────────────────────
print("=" * 72)
print(f"{'Model':<24} {'R1-F1':>8} {'R1-P':>8} {'R1-R':>8} {'R2-F1':>8} {'RL-F1':>8}")
print("=" * 72)
for name, sc in results.items():
    print(f"{name:<24} "
          f"{sc['rouge1']['f1']:>8.4f} "
          f"{sc['rouge1']['precision']:>8.4f} "
          f"{sc['rouge1']['recall']:>8.4f} "
          f"{sc['rouge2']['f1']:>8.4f} "
          f"{sc['rougeL']['f1']:>8.4f}")
print("=" * 72)
print("Higher is better (max = 1.0). R1=ROUGE-1, R2=ROUGE-2, RL=ROUGE-L")

In [ ]:

# COMPRESSION RATIO
#   = len(summary) / len(article)   → lower = more compressed
#
# NOVEL BIGRAM RATE  (faithfulness proxy)
#   = fraction of summary bigrams NOT found in source article
#   Extractive → ~0.0  (all words copied)
#   Abstractive → 0.2–0.6 (generates new phrases)
#   High novel bigrams = more abstractive but higher hallucination risk


def compression_ratio(originals, generated):
    return [len(g.split()) / max(len(o.split()), 1)
            for o, g in zip(originals, generated)]

def novel_bigram_rate(summary: str, article: str) -> float:
    def bigrams(text):
        toks = word_tokenize(text.lower())
        return set(zip(toks, toks[1:]))
    s_bg = bigrams(summary)
    a_bg = bigrams(article)
    if not s_bg:
        return 0.0
    return len(s_bg - a_bg) / len(s_bg)

def avg_sent_len(text: str) -> float:
    sents = sent_tokenize(text)
    return np.mean([len(s.split()) for s in sents]) if sents else 0.0


cr_tfidf = compression_ratio(articles, tfidf_summaries)
cr_bart  = compression_ratio(articles, bart_summaries)
cr_t5    = compression_ratio(articles, t5_summaries)
cr_ref   = compression_ratio(articles, summaries)

nbr = {
    "TF-IDF"   : np.mean([novel_bigram_rate(s, a) for s, a in zip(tfidf_summaries, articles)]),
    "BART"     : np.mean([novel_bigram_rate(s, a) for s, a in zip(bart_summaries,  articles)]),
    "T5"       : np.mean([novel_bigram_rate(s, a) for s, a in zip(t5_summaries,    articles)]),
    "Reference": np.mean([novel_bigram_rate(s, a) for s, a in zip(summaries,       articles)]),
}

asl = {
    "TF-IDF"   : np.mean([avg_sent_len(s) for s in tfidf_summaries]),
    "BART"     : np.mean([avg_sent_len(s) for s in bart_summaries]),
    "T5"       : np.mean([avg_sent_len(s) for s in t5_summaries]),
    "Reference": np.mean([avg_sent_len(s) for s in summaries]),
}

timing = {
    "TF-IDF (Extractive)": tfidf_time,
    "BART (Abstractive)" : bart_time,
    "T5 (Abstractive)"   : t5_time,
}

print("=" * 52)
print("   COMPRESSION RATIO  (lower = more compressed)")
print("=" * 52)
for name, vals in [("Reference", cr_ref), ("TF-IDF", cr_tfidf),
                   ("BART", cr_bart), ("T5", cr_t5)]:
    print(f"  {name:<12} : {np.mean(vals):.4f}  (±{np.std(vals):.4f})")

print("\n" + "=" * 52)
print("   NOVEL BIGRAM RATE  (higher = more abstractive)")
print("=" * 52)
for name, val in nbr.items():
    bar = "█" * int(val * 40)
    print(f"  {name:<12} : {val:.4f}  {bar}")

print("\n" + "=" * 52)
print("   AVG SENTENCE LENGTH  (fluency proxy)")
print("=" * 52)
for name, val in asl.items():
    print(f"  {name:<12} : {val:.1f} words/sentence")

print("\n" + "=" * 52)
print("   INFERENCE SPEED")
print("=" * 52)
for name, t in timing.items():
    print(f"  {name:<24}: {t:.1f}s total  |  {t/len(articles)*1000:.0f} ms/sample")

In [ ]:

plt.style.use('seaborn-v0_8-whitegrid')

models     = list(results.keys())
r1f1       = [results[m]['rouge1']['f1'] for m in models]
r2f1       = [results[m]['rouge2']['f1'] for m in models]
rLf1       = [results[m]['rougeL']['f1'] for m in models]
PALETTE    = ['#2196F3', '#FF5722', '#4CAF50']

fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    "Comparative Evaluation: Extractive vs Abstractive Summarization\n"
    f"CNN/DailyMail | N={CONFIG['num_samples']} samples",
    fontsize=14, fontweight='bold', y=0.98
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. ROUGE grouped bar ──────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
x   = np.arange(len(models)); w = 0.25
b1  = ax1.bar(x - w, r1f1, w, label='ROUGE-1', color='#42A5F5', alpha=0.9)
b2  = ax1.bar(x,     r2f1, w, label='ROUGE-2', color='#EF5350', alpha=0.9)
b3  = ax1.bar(x + w, rLf1, w, label='ROUGE-L', color='#66BB6A', alpha=0.9)
for bars in [b1, b2, b3]:
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.004,
                 f'{bar.get_height():.3f}', ha='center', fontsize=7.5)
ax1.set_xticks(x); ax1.set_xticklabels(models, fontsize=9)
ax1.set_ylim(0, 0.75); ax1.set_ylabel("F1 Score")
ax1.set_title("ROUGE F1 Scores by Model", fontweight='bold')
ax1.legend(fontsize=9)

# ── 2. Radar chart ────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2], polar=True)
cats   = ['R1-F1','R1-P','R1-R','R2-F1','RL-F1']
N      = len(cats)
angles = [n / N * 2 * np.pi for n in range(N)] + [0]
ax2.set_xticks(angles[:-1]); ax2.set_xticklabels(cats, size=8)
ax2.set_ylim(0, 0.75)
for idx, (name, col) in enumerate(zip(models, PALETTE)):
    sc  = results[name]
    vals = [sc['rouge1']['f1'], sc['rouge1']['precision'],
            sc['rouge1']['recall'], sc['rouge2']['f1'], sc['rougeL']['f1']] + \
           [sc['rouge1']['f1']]
    ax2.plot(angles, vals, 'o-', lw=1.5, color=col,
             label=name.split(' ')[0])
    ax2.fill(angles, vals, alpha=0.08, color=col)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=7)
ax2.set_title("ROUGE Radar", fontweight='bold', pad=15)

# ── 3. Compression boxplot ────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
bp  = ax3.boxplot(
    [cr_tfidf, cr_bart, cr_t5, cr_ref],
    labels=['TF-IDF','BART','T5','Reference'],
    patch_artist=True, notch=False
)
colors_box = ['#42A5F5','#EF5350','#66BB6A','#AB47BC']
for patch, col in zip(bp['boxes'], colors_box):
    patch.set_facecolor(col); patch.set_alpha(0.7)
ax3.set_title("Compression Ratio", fontweight='bold')
ax3.set_ylabel("Summary length / Article length")

# ── 4. Novel bigram rate bar
ax4 = fig.add_subplot(gs[1, 1])
nb_names = list(nbr.keys())
nb_vals  = list(nbr.values())
nb_cols  = ['#42A5F5','#EF5350','#66BB6A','#AB47BC']
bars4    = ax4.bar(nb_names, nb_vals, color=nb_cols, alpha=0.85, edgecolor='white')
for bar, val in zip(bars4, nb_vals):
    ax4.text(bar.get_x() + bar.get_width()/2,
             val + 0.005, f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
ax4.set_title("Novel Bigram Rate\n(Abstractiveness ↑)", fontweight='bold')
ax4.set_ylabel("Fraction of novel bigrams")
ax4.set_ylim(0, max(nb_vals) * 1.3)

# ── 5. Speed horizontal bar ───────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
sp_names  = list(timing.keys())
sp_ms     = [timing[m] / len(articles) * 1000 for m in sp_names]
bars5 = ax5.barh(sp_names, sp_ms, color=PALETTE, alpha=0.85, edgecolor='white')
for bar, val in zip(bars5, sp_ms):
    ax5.text(val + max(sp_ms)*0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.0f} ms', va='center', fontsize=9, fontweight='bold')
ax5.set_title("Inference Speed (ms/sample)", fontweight='bold')
ax5.set_xlabel("Milliseconds per sample")

plt.savefig("results_visualization.png", dpi=180,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Saved: results_visualization.png")

In [ ]:
# ── Results summary DataFrame ─────────────────────────────
rows = []
for name, sc in results.items():
    cr_map = {
        "TF-IDF (Extractive)": cr_tfidf,
        "BART (Abstractive)" : cr_bart,
        "T5 (Abstractive)"   : cr_t5,
    }
    nbr_map = {
        "TF-IDF (Extractive)": nbr["TF-IDF"],
        "BART (Abstractive)" : nbr["BART"],
        "T5 (Abstractive)"   : nbr["T5"],
    }
    rows.append({
        "Model"             : name,
        "ROUGE-1 F1"        : round(sc['rouge1']['f1'],        4),
        "ROUGE-1 Precision" : round(sc['rouge1']['precision'], 4),
        "ROUGE-1 Recall"    : round(sc['rouge1']['recall'],    4),
        "ROUGE-2 F1"        : round(sc['rouge2']['f1'],        4),
        "ROUGE-L F1"        : round(sc['rougeL']['f1'],        4),
        "Compression Ratio" : round(np.mean(cr_map[name]),     4),
        "Novel Bigram Rate" : round(nbr_map[name],             4),
        "Inference Time (s)": round(timing[name],              2),
        "ms per sample"     : round(timing[name]/len(articles)*1000, 1),
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))
df_results.to_csv("summarization_results.csv", index=False)

# ── All generated summaries ───────────────────────────────
df_summaries = pd.DataFrame({
    "article"          : articles,
    "reference_summary": summaries,
    "tfidf_summary"    : tfidf_summaries,
    "bart_summary"     : bart_summaries,
    "t5_summary"       : t5_summaries,
})
df_summaries.to_csv("all_summaries.csv", index=False)





In [ ]:


print("=" * 80)
print("       QUALITATIVE COMPARISON — 3 SAMPLE ARTICLES")
print("=" * 80)

for idx in range(3):
    print(f"\n{'─'*80}")
    print(f"  SAMPLE {idx+1}")
    print(f"{'─'*80}")
    print(f"\n ARTICLE (first 400 chars):\n{articles[idx][:400]}...\n")
    print(f" REFERENCE SUMMARY:\n{summaries[idx]}\n")
    print(f" TF-IDF EXTRACTIVE:\n{tfidf_summaries[idx]}\n")
    print(f" BART ABSTRACTIVE:\n{bart_summaries[idx]}\n")
    print(f" T5 ABSTRACTIVE:\n{t5_summaries[idx]}\n")